In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import warnings

warnings.filterwarnings('ignore')

# Device selection
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: Tesla P100-PCIE-16GB


In [ ]:

# Load data
print("Loading data...")
train_df = pd.read_csv('./dataset/train.csv')
test_df = pd.read_csv('./dataset/test.csv')

print(f"Training set: {train_df.shape}")
print(f"Test set: {test_df.shape}")

# Feature and target columns
feature_cols = [f'feature_{i}' for i in range(1, 31)]
target_cols = ['target_short', 'target_medium', 'target_long']
TARGET_WEIGHTS = {'short': 0.5, 'medium': 0.3, 'long': 0.2}

print(f"Features: {len(feature_cols)}")
print(f"Targets: {target_cols}")
print(f"Weights: {TARGET_WEIGHTS}")

Loading data...
Training set: (139392, 34)
Test set: (34348, 31)
Features: 30
Targets: ['target_short', 'target_medium', 'target_long']
Weights: {'short': 0.5, 'medium': 0.3, 'long': 0.2}


In [3]:
# Fill NaN with median and clip extreme values
for col in feature_cols:
    median_val = train_df[col].median()
    train_df[col].fillna(median_val, inplace=True)
    test_df[col].fillna(median_val, inplace=True)

    # Clip extreme values at 1% and 99% quantiles
    lower = train_df[col].quantile(0.01)
    upper = train_df[col].quantile(0.99)
    train_df[col] = train_df[col].clip(lower, upper)
    test_df[col] = test_df[col].clip(lower, upper)

In [4]:
split_idx = int(len(train_df) * 0.8)

X_train = train_df[feature_cols].iloc[:split_idx].values
y_train = train_df[target_cols].iloc[:split_idx].values
X_val = train_df[feature_cols].iloc[split_idx:].values
y_val = train_df[target_cols].iloc[split_idx:].values
X_test = test_df[feature_cols].values

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Train: (111513, 30), Val: (27879, 30), Test: (34348, 30)


In [5]:
print("="*80)
print("Training XGBoost Models")
print("="*80)

xgb_models = {}
xgb_val_predictions = {}

for i, target_name in enumerate(['short', 'medium', 'long']):
    print(f"\n--- Training XGBoost for target_{target_name} ---")
    
    # XGBoost parameters with early stopping
    xgb_params = {
        'objective': 'reg:squarederror',
        'tree_method': 'hist',
        'device': 'cuda' if torch.cuda.is_available() else 'cpu',
        'max_depth': 4,
        'learning_rate': 0.01,
        'n_estimators': 100,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'random_state': 42,
        'eval_metric': 'mae',
        'early_stopping_rounds': 50
    }
    
    model = xgb.XGBRegressor(**xgb_params)
    
    # Train
    model.fit(
        X_train, y_train[:, i],
        eval_set=[(X_val, y_val[:, i])],
        verbose=100
    )
    
    # Predict on validation set
    val_pred = model.predict(X_val)
    val_mae = mean_absolute_error(y_val[:, i], val_pred)
    
    print(f"Best iteration: {model.best_iteration}")
    print(f"Validation MAE: {val_mae:.6f}")
    
    # Store model and predictions
    xgb_models[target_name] = model
    xgb_val_predictions[target_name] = val_pred

# Calculate weighted MAE for XGBoost
xgb_weighted_mae = (
    TARGET_WEIGHTS['short'] * mean_absolute_error(y_val[:, 0], xgb_val_predictions['short']) +
    TARGET_WEIGHTS['medium'] * mean_absolute_error(y_val[:, 1], xgb_val_predictions['medium']) +
    TARGET_WEIGHTS['long'] * mean_absolute_error(y_val[:, 2], xgb_val_predictions['long'])
)

print("\n" + "="*80)
print("XGBoost Summary")
print("="*80)
print(f"Short MAE:  {mean_absolute_error(y_val[:, 0], xgb_val_predictions['short']):.6f} (weight: 0.5)")
print(f"Medium MAE: {mean_absolute_error(y_val[:, 1], xgb_val_predictions['medium']):.6f} (weight: 0.3)")
print(f"Long MAE:   {mean_absolute_error(y_val[:, 2], xgb_val_predictions['long']):.6f} (weight: 0.2)")
print(f"Weighted MAE: {xgb_weighted_mae:.6f}")
print("="*80)

Training XGBoost Models

--- Training XGBoost for target_short ---
[0]	validation_0-mae:0.00286
[50]	validation_0-mae:0.00286
Best iteration: 1
Validation MAE: 0.002856

--- Training XGBoost for target_medium ---
[0]	validation_0-mae:0.00748
[49]	validation_0-mae:0.00801
Best iteration: 0
Validation MAE: 0.007485

--- Training XGBoost for target_long ---
[0]	validation_0-mae:0.01616
[64]	validation_0-mae:0.01647
Best iteration: 14
Validation MAE: 0.016102

XGBoost Summary
Short MAE:  0.002856 (weight: 0.5)
Medium MAE: 0.007485 (weight: 0.3)
Long MAE:   0.016102 (weight: 0.2)
Weighted MAE: 0.006894


In [6]:
# Standardize features for neural network (don't scale targets)
print("Standardizing features...")
from sklearn.preprocessing import RobustScaler

scaler_X = RobustScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_val_scaled = scaler_X.transform(X_val)
X_test_scaled = scaler_X.transform(X_test)

print("Standardization complete")

class MultiTargetDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y) if y is not None else None
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]

# Create datasets (use original y_train, y_val - no scaling)
train_dataset = MultiTargetDataset(X_train_scaled, y_train)
val_dataset = MultiTargetDataset(X_val_scaled, y_val)
test_dataset = MultiTargetDataset(X_test_scaled)

# Create dataloaders
BATCH_SIZE = 512*8*8

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

class MLPMultiTarget(nn.Module):
    def __init__(self, input_dim, hidden_dims=[128, 64], num_targets=3, dropout=0.2):
        super(MLPMultiTarget, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = hidden_dim
        
        self.shared_layers = nn.Sequential(*layers)
        
        # Output layer: predict all 3 targets simultaneously
        self.output_layer = nn.Linear(prev_dim, num_targets)
    
    def forward(self, x):
        x = self.shared_layers(x)
        return self.output_layer(x)

# Initialize model
INPUT_DIM = X_train_scaled.shape[1]
mlp_model = MLPMultiTarget(input_dim=INPUT_DIM).to(DEVICE)

print(f"MLP Model:")
print(mlp_model)
print(f"\nTotal parameters: {sum(p.numel() for p in mlp_model.parameters()):,}")

Standardizing features...
Standardization complete
Train batches: 4
Validation batches: 1
Test batches: 2
MLP Model:
MLPMultiTarget(
  (shared_layers): Sequential(
    (0): Linear(in_features=30, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
  )
  (output_layer): Linear(in_features=64, out_features=3, bias=True)
)

Total parameters: 12,803


In [7]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        
        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)

def eval_epoch(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in loader:
            if len(batch) == 2:
                X_batch, y_batch = batch
                X_batch = X_batch.to(device)
                
                outputs = model(X_batch)
                
                all_preds.append(outputs.cpu().numpy())
                all_labels.append(y_batch.numpy())
            else:
                X_batch = batch.to(device)
                outputs = model(X_batch)
                all_preds.append(outputs.cpu().numpy())
    
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels) if len(all_labels) > 0 else None
    
    return all_preds, all_labels

print("Training functions defined")

Training functions defined


In [8]:
print("Training MLP model...")
print("="*80)

criterion = nn.L1Loss()  # Use MAE loss directly
optimizer = optim.AdamW(mlp_model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

EPOCHS = 20
best_weighted_mae = float('inf')
patience = 5
patience_counter = 0

for epoch in range(EPOCHS):
    train_loss = train_epoch(mlp_model, train_loader, criterion, optimizer, DEVICE)
    val_preds, val_labels = eval_epoch(mlp_model, val_loader, DEVICE)
    
    # Calculate MAE for each target
    mae_short = mean_absolute_error(val_labels[:, 0], val_preds[:, 0])
    mae_medium = mean_absolute_error(val_labels[:, 1], val_preds[:, 1])
    mae_long = mean_absolute_error(val_labels[:, 2], val_preds[:, 2])
    
    # Calculate weighted MAE
    weighted_mae = (
        TARGET_WEIGHTS['short'] * mae_short +
        TARGET_WEIGHTS['medium'] * mae_medium +
        TARGET_WEIGHTS['long'] * mae_long
    )
    
    old_lr = optimizer.param_groups[0]['lr']
    scheduler.step(weighted_mae)
    new_lr = optimizer.param_groups[0]['lr']
    
    if new_lr != old_lr:
        print(f"  Learning rate reduced: {old_lr:.6f} -> {new_lr:.6f}")
    
    if weighted_mae < best_weighted_mae:
        best_weighted_mae = weighted_mae
        patience_counter = 0
    else:
        patience_counter += 1
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS}:")
        print(f"  Train Loss: {train_loss:.6f}")
        print(f"  Val Weighted MAE: {weighted_mae:.6f} (Best: {best_weighted_mae:.6f})")
        print(f"    Short: {mae_short:.6f}, Medium: {mae_medium:.6f}, Long: {mae_long:.6f}")
    
    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

print(f"\nMLP training complete")
print(f"Best validation Weighted MAE: {best_weighted_mae:.6f}")

Training MLP model...
Epoch 10/20:
  Train Loss: 0.068438
  Val Weighted MAE: 0.021922 (Best: 0.021922)
    Short: 0.020169, Medium: 0.020286, Long: 0.028761
Epoch 20/20:
  Train Loss: 0.033790
  Val Weighted MAE: 0.009795 (Best: 0.009795)
    Short: 0.007051, Medium: 0.009271, Long: 0.017439

MLP training complete
Best validation Weighted MAE: 0.009795


In [9]:
val_preds, val_labels = eval_epoch(mlp_model, val_loader, DEVICE)

mlp_weighted_mae = (
    TARGET_WEIGHTS['short'] * mean_absolute_error(val_labels[:, 0], val_preds[:, 0]) +
    TARGET_WEIGHTS['medium'] * mean_absolute_error(val_labels[:, 1], val_preds[:, 1]) +
    TARGET_WEIGHTS['long'] * mean_absolute_error(val_labels[:, 2], val_preds[:, 2])
)

print("="*80)
print("MLP Summary")
print("="*80)
print(f"Short MAE:  {mean_absolute_error(val_labels[:, 0], val_preds[:, 0]):.6f} (weight: 0.5)")
print(f"Medium MAE: {mean_absolute_error(val_labels[:, 1], val_preds[:, 1]):.6f} (weight: 0.3)")
print(f"Long MAE:   {mean_absolute_error(val_labels[:, 2], val_preds[:, 2]):.6f} (weight: 0.2)")
print(f"Weighted MAE: {mlp_weighted_mae:.6f}")
print("="*80)

MLP Summary
Short MAE:  0.007051 (weight: 0.5)
Medium MAE: 0.009271 (weight: 0.3)
Long MAE:   0.017439 (weight: 0.2)
Weighted MAE: 0.009795


In [10]:
# Compare models
comparison = pd.DataFrame({
    'Model': ['XGBoost (3 models)', 'MLP (multi-target)'],
    'Weighted MAE': [xgb_weighted_mae, mlp_weighted_mae]
})

print(comparison.to_string(index=False))

best_model = 'XGBoost' if xgb_weighted_mae < mlp_weighted_mae else 'MLP'
print(f"\nBest model: {best_model}")

             Model  Weighted MAE
XGBoost (3 models)      0.006894
MLP (multi-target)      0.009795

Best model: XGBoost


In [11]:
print("Generating XGBoost predictions...")
test_pred_xgb_short = xgb_models['short'].predict(X_test)
test_pred_xgb_medium = xgb_models['medium'].predict(X_test)
test_pred_xgb_long = xgb_models['long'].predict(X_test)

submission_xgb = pd.DataFrame({
    'id': test_df['id'],
    'target_short': test_pred_xgb_short,
    'target_medium': test_pred_xgb_medium,
    'target_long': test_pred_xgb_long
})
submission_xgb.to_csv('submission.csv', index=False)
print(f"✅ XGBoost submission saved: submission_xgb.csv")

# print("\nGenerating MLP predictions...")
# test_preds_mlp, _ = eval_epoch(mlp_model, test_loader, DEVICE)
# submission_mlp = pd.DataFrame({
#     'id': test_df['id'],
#     'target_short': test_preds_mlp[:, 0],
#     'target_medium': test_preds_mlp[:, 1],
#     'target_long': test_preds_mlp[:, 2]
# })
# submission_mlp.to_csv('submission.csv', index=False)
# print(f"✅ MLP submission saved: submission_mlp.csv")

Generating XGBoost predictions...
✅ XGBoost submission saved: submission_xgb.csv
